Análise exploratória dos arquivos originais com PySpark

Exploração para conhecer os dados antes da criação das camadas Bronze, Silver e Gold.

In [17]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, mean, when

In [18]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('analise_exploratoria_alfabetizacao')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print('Versão do Spark:', spark.version)

Versão do Spark: 3.5.9


In [19]:
pasta_atual = Path.cwd()

if pasta_atual.name == 'notebooks':
    raiz_projeto = pasta_atual.parent
else:
    raiz_projeto = pasta_atual

pasta_raw = raiz_projeto / 'data' / 'arquivos_raw'

print('Raiz do projeto:', raiz_projeto)
print('Pasta dos arquivos originais:', pasta_raw)
print('A pasta existe?', pasta_raw.exists())

Raiz do projeto: c:\Users\claud\Documents\Py\tech_challenge_02
Pasta dos arquivos originais: c:\Users\claud\Documents\Py\tech_challenge_02\data\arquivos_raw
A pasta existe? True


In [20]:
arquivos_encontrados = sorted(pasta_raw.glob('*.csv.gz'))

for arquivo in arquivos_encontrados:
    tamanho_kb = arquivo.stat().st_size / 1024
    print(arquivo.name, '-', round(tamanho_kb, 2), 'KB')

print('Total de arquivos:', len(arquivos_encontrados))

br_inep_avaliacao_alfabetizacao_aluno.csv.gz - 67627.0 KB
br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv.gz - 0.17 KB
br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv.gz - 137.37 KB
br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv.gz - 0.98 KB
br_inep_avaliacao_alfabetizacao_municipio.csv.gz - 359.34 KB
br_inep_avaliacao_alfabetizacao_uf.csv.gz - 3.03 KB
Total de arquivos: 6


In [21]:
def carregar_arquivo(nome_arquivo):
    caminho = pasta_raw / nome_arquivo

    dataframe = (
        spark.read
        .option('header', True)
        .option('inferSchema', True)
        .option('encoding', 'UTF-8')
        .csv(caminho.as_posix())
    )

    return dataframe

In [22]:
df_uf = carregar_arquivo('br_inep_avaliacao_alfabetizacao_uf.csv.gz')
df_meta_brasil = carregar_arquivo('br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv.gz')
df_meta_uf = carregar_arquivo('br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv.gz')
df_meta_municipio = carregar_arquivo('br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv.gz')
df_municipio = carregar_arquivo('br_inep_avaliacao_alfabetizacao_municipio.csv.gz')
df_alunos = carregar_arquivo('br_inep_avaliacao_alfabetizacao_aluno.csv.gz')

print('DataFrames carregados com sucesso.')

DataFrames carregados com sucesso.


Quantidade de registros e colunas

In [23]:
dataframes = {
    'uf': df_uf,
    'meta_brasil': df_meta_brasil,
    'meta_uf': df_meta_uf,
    'meta_municipio': df_meta_municipio,
    'municipio': df_municipio,
    'alunos': df_alunos,
}

for nome, dataframe in dataframes.items():
    linhas = dataframe.count()
    colunas = len(dataframe.columns)
    print(nome, '- linhas:', linhas, '- colunas:', colunas)

uf - linhas: 145 - colunas: 15
meta_brasil - linhas: 3 - colunas: 11
meta_uf - linhas: 54 - colunas: 12
meta_municipio - linhas: 10704 - colunas: 13
municipio - linhas: 23995 - colunas: 15
alunos - linhas: 3867999 - colunas: 12


Primeiras linhas

In [24]:
for nome, dataframe in dataframes.items():
    print('DataFrame:', nome)
    dataframe.show(5, truncate=False)

DataFrame: uf
+----+--------+-----+----+------------------+---------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+
|ano |sigla_uf|serie|rede|taxa_alfabetizacao|media_portugues|proporcao_aluno_nivel_0|proporcao_aluno_nivel_1|proporcao_aluno_nivel_2|proporcao_aluno_nivel_3|proporcao_aluno_nivel_4|proporcao_aluno_nivel_5|proporcao_aluno_nivel_6|proporcao_aluno_nivel_7|proporcao_aluno_nivel_8|
+----+--------+-----+----+------------------+---------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+
|2023|AM      |2    |3   |49.2              |733.6637       |NULL                   |NULL                   |NULL                   |NULL               

Esquema das tabelas

In [ ]:
for nome, dataframe in dataframes.items():
    print('Esquema:', nome)
    dataframe.printSchema()

Valores ausentes

In [26]:
def mostrar_valores_nulos(dataframe):
    contagens = []

    for nome_coluna in dataframe.columns:
        contagem = count(when(col(nome_coluna).isNull(), nome_coluna)).alias(nome_coluna)
        contagens.append(contagem)

    dataframe.select(contagens).show(truncate=False)

In [ ]:
for nome, dataframe in dataframes.items():
    print('Valores nulos:', nome)
    mostrar_valores_nulos(dataframe)

Linhas duplicadas

In [27]:
for nome, dataframe in dataframes.items():
    total = dataframe.count()
    total_sem_repeticao = dataframe.dropDuplicates().count()
    duplicados = total - total_sem_repeticao
    print(nome, '- linhas duplicadas:', duplicados)

uf - linhas duplicadas: 0
meta_brasil - linhas duplicadas: 0
meta_uf - linhas duplicadas: 0
meta_municipio - linhas duplicadas: 0
municipio - linhas duplicadas: 0
alunos - linhas duplicadas: 0


Estatísticas dos resultados

In [25]:
print('Estatísticas por UF:')
df_uf.select('taxa_alfabetizacao', 'media_portugues').summary().show()

print('Estatísticas por município:')
df_municipio.select('taxa_alfabetizacao', 'media_portugues').summary().show()

Estatísticas por UF:
+-------+------------------+------------------+
|summary|taxa_alfabetizacao|   media_portugues|
+-------+------------------+------------------+
|  count|               145|               145|
|   mean| 56.37400000000001| 745.3366400000003|
| stddev|13.190042035144877|15.785106194528892|
|    min|             30.57|           712.562|
|    25%|             47.45|          734.2547|
|    50%|             55.87|          744.8152|
|    75%|             63.55|             753.3|
|    max|             86.21|            797.34|
+-------+------------------+------------------+

Estatísticas por município:
+-------+------------------+------------------+
|summary|taxa_alfabetizacao|   media_portugues|
+-------+------------------+------------------+
|  count|             23995|             23995|
|   mean| 61.44454511356532| 751.3757078891408|
| stddev|19.743520505811617|23.232836703593907|
|    min|              2.12|          673.2983|
|    25%|             47.17|          

Distribuição por ano e rede

In [28]:
(
    df_uf
    .groupBy('ano', 'rede')
    .agg(
        count('*').alias('quantidade_registros'),
        mean('taxa_alfabetizacao').alias('media_taxa_alfabetizacao'),
    )
    .orderBy('ano', 'rede')
    .show(truncate=False)
)

+----+----+--------------------+------------------------+
|ano |rede|quantidade_registros|media_taxa_alfabetizacao|
+----+----+--------------------+------------------------+
|2023|2   |22                  |57.88681818181817       |
|2023|3   |24                  |54.032916666666665      |
|2023|5   |24                  |54.25125                |
|2024|0   |1                   |35.96                   |
|2024|2   |24                  |60.134166666666665      |
|2024|3   |25                  |56.296800000000005      |
|2024|5   |25                  |56.612000000000016      |
+----+----+--------------------+------------------------+



Visualização das metas

In [29]:
print('Metas do Brasil:')
df_meta_brasil.show(truncate=False)

print('Exemplo das metas por UF:')
df_meta_uf.show(10, truncate=False)

print('Exemplo das metas por município:')
df_meta_municipio.show(10, truncate=False)

Metas do Brasil:
+----+-------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+
|ano |rede   |taxa_alfabetizacao|meta_alfabetizacao_2024|meta_alfabetizacao_2025|meta_alfabetizacao_2026|meta_alfabetizacao_2027|meta_alfabetizacao_2028|meta_alfabetizacao_2029|meta_alfabetizacao_2030|percentual_participacao|
+----+-------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+
|2025|Pública|66.0              |60.0                   |64.0                   |67.0                   |71.0                   |74.0                   |77.0                   |80                     |88.0                   |
|2024|Pública|59.2              |59.9                   |63.77                 

Exploração dos alunos

In [30]:
print('Quantidade por presença:')
df_alunos.groupBy('presenca').count().orderBy('presenca').show()

print('Quantidade por situação de alfabetização:')
df_alunos.groupBy('alfabetizado').count().orderBy('alfabetizado').show()

print('Quantidade por rede:')
df_alunos.groupBy('rede').count().orderBy('rede').show()

Quantidade por presença:
+--------+-------+
|presenca|  count|
+--------+-------+
|       0| 512153|
|       1|3355846|
+--------+-------+

Quantidade por situação de alfabetização:
+------------+-------+
|alfabetizado|  count|
+------------+-------+
|           0|1883453|
|           1|1984546|
+------------+-------+

Quantidade por rede:
+----+-------+
|rede|  count|
+----+-------+
|   2| 435398|
|   3|3432576|
|   4|     25|
+----+-------+



Estatísticas dos alunos


In [31]:
df_alunos.select('proficiencia', 'peso_aluno').summary().show()

+-------+-----------------+------------------+
|summary|     proficiencia|        peso_aluno|
+-------+-----------------+------------------+
|  count|          3354661|           3354661|
|   mean| 748.379854020743|1.1484895204979162|
| stddev|47.58597537092779|0.3585159781550361|
|    min|           578.46|         0.0952381|
|    25%|           719.75|         1.0006768|
|    50%|           753.15|         1.0916443|
|    75%|      779.6566085|         1.2085876|
|    max|       904.382031|          142.5454|
+-------+-----------------+------------------+



In [32]:
spark.stop()